In [ ]:
# CELL 1 — ComfyUI + Manager + Kaggle MultiGPU fork
import os, sys, subprocess, pathlib, torch
COMFY = pathlib.Path("/kaggle/working/ComfyUI")
CUSTOM = COMFY / "custom_nodes"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)
print("PyTorch", torch.__version__, "CUDA", torch.version.cuda, "GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}: {p.name} {p.total_memory/1024**3:.2f} GiB")
if torch.cuda.device_count() != 2:
    raise RuntimeError("Select Kaggle GPU T4 x2 before running.")
if not COMFY.exists():
    run(["git", "clone", "https://github.com/Comfy-Org/ComfyUI.git", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", COMFY/"requirements.txt"])
CUSTOM.mkdir(parents=True, exist_ok=True)
for name, url in [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-MultiGPU", "https://github.com/MrKK4/ComfyUI-MultiGPU-Kaggle.git"),
]:
    dest = CUSTOM/name
    if not dest.exists():
        run(["git", "clone", url, dest])
    req = dest/"requirements.txt"
    if req.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "-r", req])
run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_transfer"])
print("Install complete. No ComfyUI core / comfy_aimdo files are patched.")


In [ ]:
# CELL 2 — Download models and convert your original workflow to the Dual-T4 profile
import os, sys, pathlib, json, shutil
from huggingface_hub import hf_hub_download
COMFY = pathlib.Path("/kaggle/working/ComfyUI")
MODELS = COMFY/"models"
HF_CACHE = pathlib.Path("/kaggle/working/hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    token = token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
def fetch(repo, filename, folder):
    d = MODELS/folder
    d.mkdir(parents=True, exist_ok=True)
    dst = d/pathlib.Path(filename).name
    if dst.exists() and dst.stat().st_size > 100_000_000:
        print("Already present:", dst); return
    cached = pathlib.Path(hf_hub_download(repo_id=repo, filename=filename, cache_dir=str(HF_CACHE), token=token or None))
    if dst.exists() or dst.is_symlink(): dst.unlink()
    dst.symlink_to(cached.resolve())
    print("Installed", dst)
fetch("Lightricks/LTX-2.3-fp8", "ltx-2.3-22b-dev-fp8.safetensors", "checkpoints")
fetch("Comfy-Org/ltx-2.3", "split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors", "loras")
fetch("Comfy-Org/ltx-2", "split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors", "text_encoders")
fetch("Comfy-Org/ltx-2", "split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors", "loras")
fetch("Lightricks/LTX-2.3", "ltx-2.3-spatial-upscaler-x2-1.1.safetensors", "latent_upscale_models")
# Add your original video_ltx2_3_ia2v_Dual_gpu.json to a Kaggle Dataset first.
candidates = list(pathlib.Path("/kaggle/input").rglob("video_ltx2_3_ia2v_Dual_gpu.json"))
if not candidates:
    raise FileNotFoundError("Add video_ltx2_3_ia2v_Dual_gpu.json to a Kaggle Dataset, then rerun Cell 2.")
wf = json.loads(candidates[0].read_text(encoding="utf-8"))
sg = wf["definitions"]["subgraphs"][0]
outer = next(n for n in wf["nodes"] if n.get("id") == 340)
outer["widgets_values"][1] = False   # prompt enhancer off
outer["widgets_values"][2] = 768
outer["widgets_values"][3] = 512
outer["widgets_values"][5] = 4       # 4-second profile
# Main checkpoint distributed across both T4s, with overflow in CPU RAM.
loader = next(n for n in sg["nodes"] if n.get("id") == 317)
loader["type"] = "CheckpointLoaderSimpleDisTorch2MultiGPU"
loader.setdefault("properties", {})["Node name for S&R"] = "CheckpointLoaderSimpleDisTorch2MultiGPU"
loader["properties"]["cnr_id"] = "ComfyUI-MultiGPU-Kaggle"
loader["inputs"] = [i for i in loader.get("inputs", []) if i.get("name") == "ckpt_name"]
for name, typ in [("compute_device","COMBO"),("virtual_vram_gb","FLOAT"),("donor_device","COMBO"),("expert_mode_allocations","STRING"),("eject_models","BOOLEAN")]:
    loader["inputs"].append({"localized_name":name,"name":name,"type":typ,"widget":{"name":name},"link":None})
loader["widgets_values"] = ["ltx-2.3-22b-dev-fp8.safetensors","cuda:0",0.0,"cpu","cuda:0,7gb;cuda:1,7gb;cpu,*",True]
loader["widgets_values_named"] = {"ckpt_name":"ltx-2.3-22b-dev-fp8.safetensors","compute_device":"cuda:0","virtual_vram_gb":0.0,"donor_device":"cpu","expert_mode_allocations":"cuda:0,7gb;cuda:1,7gb;cpu,*","eject_models":True}
# Keep stock tiled VAE decode; do not introduce an unverified custom VAE socket layout.
out = COMFY/"user/default/workflows/LTX23_AI2V_DualT4_Optimized.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(wf, ensure_ascii=False, indent=2), encoding="utf-8")
print("Optimized workflow saved:", out)
print("Profile: 768x512, 4s; 7GB model blocks on each GPU, remainder CPU.")
print("Accept the Hugging Face model terms and set Kaggle Secret HF_TOKEN if access is denied.")


In [ ]:
# CELL 3 — Launch ComfyUI
import os, sys, pathlib, subprocess, time, urllib.request
COMFY = pathlib.Path("/kaggle/working/ComfyUI")
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
log = open("/kaggle/working/comfyui.log", "w", buffering=1)
cmd = [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188", "--reserve-vram", "1.5", "--disable-cuda-malloc"]
proc = subprocess.Popen(cmd, cwd=COMFY, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
print("PID", proc.pid, "log /kaggle/working/comfyui.log")
ready = False
for _ in range(90):
    if proc.poll() is not None:
        print("\n".join(pathlib.Path("/kaggle/working/comfyui.log").read_text(errors="ignore").splitlines()[-80:]))
        raise RuntimeError("ComfyUI exited; see log.")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
            if r.status == 200: ready = True; break
    except Exception: pass
    time.sleep(2)
print("ComfyUI ready at http://127.0.0.1:8188" if ready else "Not ready; inspect comfyui.log")
